# Train a PyTorch Classifier with Minibatches

This tutorial shows how to train a neural-network classifier using expression
minibatches streamed from an Atlas. It follows the same label-alignment
principles introduced in
{doc}`train-logistic-regression-with-minibatches`, while replacing the
incremental scikit-learn estimator with a custom PyTorch training loop.

The complete cell-by-gene matrix remains in the Atlas. Only the current
expression minibatch, model parameters, and target labels need to be available
during training.

By the end of this tutorial, you will be able to:

- define a labeled training population in `obs`;
- align labels with deterministic expression minibatches;
- train a PyTorch model for several streaming passes;
- move minibatches between CPU and GPU safely;
- evaluate the training pipeline without retaining all expression data;
- save the model together with its class and feature definitions.

## Before You Begin

This tutorial assumes that:

- quality control and preprocessing have been completed;
- `obs.cell_type_manual` contains the target labels;
- scaled expression values are available in `data_scale`;
- PyTorch is installed;
- each dense expression minibatch fits in host memory and, when applicable,
  device memory.

Open the existing Atlas:



In [ ]:
import os
from pathlib import Path
import numpy as np
import scatlaspy as sap

os.chdir(Path("~/scAtlaspy-code-analysis").expanduser())

atlas_path = Path("./tmp/tutorials/basic_pbmc3k/pbmc3k_basic_copy.sasql")

if not atlas_path.is_file():
    raise FileNotFoundError(f"Atlas database not found: {atlas_path}")

atlas = sap.Atlas(
    atlas_path,
    db_memory_limit="8GB",
)



```{note}
Cell-type labels are used here as an example supervised target. The same
pattern can be adapted to another categorical outcome stored in `obs`.
```

## 1. Define the Labeled Training Population

The expression stream and label vector must contain exactly the same cells.

Create a Boolean column identifying cells that pass quality-control filtering
and have a non-missing training label:



In [ ]:
atlas.execute_sql("""
    ALTER TABLE obs
    ADD COLUMN IF NOT EXISTS model_labeled_cells BOOLEAN
""")

atlas.execute_sql("""
    UPDATE obs
    SET model_labeled_cells =
        COALESCE(filter_cells, FALSE)
        AND cell_type_manual IS NOT NULL
""")



Inspect the number of labeled cells in each class:



In [ ]:
class_counts = atlas.query("""
    SELECT
        cell_type_manual,
        COUNT(*) AS n_cells
    FROM obs
    WHERE model_labeled_cells
    GROUP BY cell_type_manual
    ORDER BY n_cells DESC
""")

class_counts



Confirm that at least two classes are present and that each class contains
enough cells for the intended experiment.

```{important}
Do not build the read index from all filtered cells and then remove unlabeled
cells only from the label table. The resulting labels would no longer align
with the expression stream.
```

## 2. Build the Training Read Index

Build a read index from the labeled population:



In [ ]:
atlas.build_read_index(
    cell_condition="model_labeled_cells",
    gene_condition="filter_genes",
    use_hvg=True,
    use_data="data_scale",
)



The model input now contains:

- cells selected by `model_labeled_cells`;
- genes selected by `filter_genes`;
- genes marked as highly variable;
- scaled expression values stored in `data_scale`.

```{important}
The trained model depends on the exact gene set, gene order, and expression
representation defined by this read index. These must be retained with the
model checkpoint.
```

## 3. Read and Encode the Labels

In `single-pass` mode, expression minibatches follow ascending
`filter_cell_id` order. Retrieve the labels in the same order:



In [ ]:
label_df = atlas.query("""
    SELECT
        filter_cell_id,
        cell_type_manual
    FROM obs
    WHERE filter_cell_id IS NOT NULL
    ORDER BY filter_cell_id
""")



Validate the label table:



In [ ]:
if label_df.empty:
    raise ValueError(
        "The current read index contains no labeled cells."
    )

if label_df["cell_type_manual"].isna().any():
    raise ValueError(
        "The current read index contains cells without labels."
    )

if label_df["filter_cell_id"].duplicated().any():
    raise ValueError(
        "The current read index contains duplicated filter_cell_id values."
    )



Encode the class labels as consecutive integers:



In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_all = label_encoder.fit_transform(
    label_df["cell_type_manual"].to_numpy()
)

classes = label_encoder.classes_
n_classes = len(classes)

if n_classes < 2:
    raise ValueError(
        "At least two classes are required for classification."
    )

print(f"Training cells: {len(y_all):,}")
print(f"Classes: {n_classes:,}")
print(classes)



The encoded label vector is retained in memory, but it is much smaller than the
complete expression matrix.

## 4. Configure PyTorch

Import PyTorch and select the compute device:



In [ ]:
import torch
import torch.nn as nn

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Using device: {device}")



Set random seeds for repeatable initialization:



In [ ]:
random_seed = 42

np.random.seed(random_seed)
torch.manual_seed(random_seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(random_seed)



```{note}
Setting random seeds improves repeatability, but exact numerical reproducibility
may still depend on the PyTorch version, hardware, and selected device
operations.
```

## 5. Define the Classifier

The number of input genes can be obtained from the first expression minibatch.
The model is therefore initialized when the first batch is encountered.

Define a small multilayer perceptron:



In [ ]:
class CellTypeClassifier(nn.Module):
    def __init__(
        self,
        n_features: int,
        n_classes: int,
        hidden_size: int = 128,
    ) -> None:
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(n_features, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, n_classes),
        )

    def forward(
        self,
        X: torch.Tensor,
    ) -> torch.Tensor:
        return self.network(X)



Initialize the training objects:



In [ ]:
model = None
optimizer = None
loss_fn = nn.CrossEntropyLoss()

hidden_size = 128
learning_rate = 1e-3



`CrossEntropyLoss` expects unnormalized class logits from the model and integer
class identifiers from `0` to `n_classes - 1`.

## 6. Train for Multiple Streaming Passes

Set the training configuration:



In [ ]:
batch_size = 2048
n_epochs = 3



Each epoch creates a new deterministic single-pass expression stream:



In [ ]:
for epoch in range(1, n_epochs + 1):
    offset = 0
    epoch_loss_sum = 0.0
    epoch_cells = 0

    if model is not None:
        model.train()

    for batch_id, X_batch in enumerate(
        atlas.get_minibatch_dense(
            pass_mode="single-pass",
            batch_size=batch_size,
        ),
        start=1,
    ):
        X_batch = np.asarray(
            X_batch,
            dtype=np.float32,
        )

        if X_batch.ndim != 2:
            raise ValueError(
                "Each expression minibatch must be a two-dimensional matrix."
            )

        n_cells, n_genes = X_batch.shape

        if model is None:
            model = CellTypeClassifier(
                n_features=n_genes,
                n_classes=n_classes,
                hidden_size=hidden_size,
            ).to(device)

            optimizer = torch.optim.Adam(
                model.parameters(),
                lr=learning_rate,
            )

            model.train()

        y_batch = y_all[offset : offset + n_cells]

        if len(y_batch) != n_cells:
            raise ValueError(
                "The label vector is shorter than the expression stream."
            )

        X_tensor = torch.as_tensor(
            X_batch,
            dtype=torch.float32,
            device=device,
        )

        y_tensor = torch.as_tensor(
            y_batch,
            dtype=torch.long,
            device=device,
        )

        optimizer.zero_grad(set_to_none=True)

        logits = model(X_tensor)
        loss = loss_fn(logits, y_tensor)

        loss.backward()
        optimizer.step()

        epoch_loss_sum += loss.item() * n_cells
        epoch_cells += n_cells
        offset += n_cells

        if batch_id == 1 or batch_id % 50 == 0:
            print(
                f"Epoch {epoch}/{n_epochs}, "
                f"batch {batch_id}: "
                f"loss={loss.item():.4f}, "
                f"processed={offset:,}"
            )

    if offset != len(y_all):
        raise ValueError(
            "The number of streamed cells does not match the label vector."
        )

    if epoch_cells == 0:
        raise ValueError(
            "The training stream contained no cells."
        )

    mean_epoch_loss = epoch_loss_sum / epoch_cells

    print(
        f"Completed epoch {epoch}/{n_epochs}: "
        f"mean loss={mean_epoch_loss:.4f}, "
        f"cells={epoch_cells:,}"
    )



Only the current dense expression minibatch and its tensor representation are
required during each update.

```{note}
Repeated `single-pass` traversal preserves the same global cell order between
epochs. This provides safe label alignment with the current public iterator,
but it does not randomly shuffle supervised observations between epochs.

A label-aware iterator that returns cell identifiers together with each
minibatch would allow randomized supervised training.
```

## 7. Check the Training Pipeline

Run another single-pass traversal and calculate training-set accuracy without
retaining all predictions:



In [ ]:
if model is None:
    raise RuntimeError(
        "The model was not initialized because no training data were read."
    )

model.eval()

correct = 0
total = 0
offset = 0

with torch.inference_mode():
    for X_batch in atlas.get_minibatch_dense(
        pass_mode="single-pass",
        batch_size=batch_size,
    ):
        X_batch = np.asarray(
            X_batch,
            dtype=np.float32,
        )

        n_cells = X_batch.shape[0]
        y_batch = y_all[offset : offset + n_cells]

        if len(y_batch) != n_cells:
            raise ValueError(
                "The label vector is shorter than the evaluation stream."
            )

        X_tensor = torch.as_tensor(
            X_batch,
            dtype=torch.float32,
            device=device,
        )

        logits = model(X_tensor)
        predictions = logits.argmax(dim=1).cpu().numpy()

        correct += int(
            np.count_nonzero(predictions == y_batch)
        )

        total += n_cells
        offset += n_cells

if offset != len(y_all):
    raise ValueError(
        "The number of evaluated cells does not match the label vector."
    )

if total == 0:
    raise ValueError(
        "The evaluation stream contained no cells."
    )

training_accuracy = correct / total

print(
    f"Training-set accuracy: "
    f"{training_accuracy:.3f}"
)



```{warning}
Training-set accuracy is useful for checking label alignment and confirming that
the training loop functions correctly. It is not an unbiased estimate of model
performance because the same cells were used for fitting and evaluation.
```

## 8. Evaluate on Held-out Cells

For a meaningful evaluation, define non-overlapping training and test
populations in `obs`, such as:

```text
model_train_cells
model_test_cells
```

Then:

1. build the read index from `model_train_cells`;
2. retrieve the training labels in its `filter_cell_id` order;
3. train the model;
4. rebuild the read index from `model_test_cells`;
5. retrieve the test labels in the new read-index order;
6. evaluate the model with a new single-pass stream.

The labels must be queried again after each read-index construction because
`filter_cell_id` describes the current analysis view.

```{tip}
For cell-atlas applications, holding out complete donors, samples, studies, or
technologies may provide a more meaningful estimate of generalization than
randomly splitting individual cells.
```

Consider metrics such as precision, recall, F1 score, balanced accuracy, and a
confusion matrix when classes are imbalanced.

## 9. Save the Model and Input Definition

Retrieve the selected genes in read-index order:



In [ ]:
feature_df = atlas.query("""
    SELECT
        filter_gene_id,
        atlas_gene_id,
        atlas_gene_name
    FROM var
    WHERE filter_gene_id IS NOT NULL
    ORDER BY filter_gene_id
""")



Verify the feature count:



In [ ]:
if len(feature_df) != model.network[0].in_features:
    raise ValueError(
        "The selected gene count does not match the model input dimension."
    )



Save the model checkpoint:



In [ ]:
from pathlib import Path

output_dir = Path("./results")
output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

checkpoint_path = output_dir / "pytorch_cell_type_classifier.pt"

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "n_features": model.network[0].in_features,
        "hidden_size": hidden_size,
        "n_classes": n_classes,
        "classes": classes.tolist(),
        "expression_field": "data_scale",
        "use_hvg": True,
        "random_seed": random_seed,
    },
    checkpoint_path,
)

feature_df.to_csv(
    output_dir / "pytorch_model_features.csv",
    index=False,
)

print(f"Saved checkpoint to {checkpoint_path}")



A reusable model requires more than its learned parameters. Retain:

- the model architecture;
- class names and their encoded order;
- selected gene names and their exact order;
- the expression field;
- normalization and scaling settings;
- cell and gene selection rules;
- training hyperparameters;
- the PyTorch and scAtlasPy versions.

## 10. Load the Saved Model

Reconstruct the model using the saved architecture metadata:



In [ ]:
checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
)

loaded_model = CellTypeClassifier(
    n_features=checkpoint["n_features"],
    n_classes=checkpoint["n_classes"],
    hidden_size=checkpoint["hidden_size"],
).to(device)

loaded_model.load_state_dict(
    checkpoint["model_state_dict"]
)

loaded_model.eval()



Before applying the loaded model, verify that the current read index contains
the same genes in the same order and uses the same expression representation as
the saved training configuration.

## Why Supervised Training Uses single-pass

The current dense minibatch iterator returns `X_batch`, but does not return the
cell identifiers associated with randomized minibatches.

A label vector ordered by `filter_cell_id` can therefore be aligned safely with
a deterministic single-pass stream:



In [ ]:
for X_batch in atlas.get_minibatch_dense(
    pass_mode="single-pass",
    batch_size=2048,
):
    ...



Do not combine an ordered label vector with:



In [ ]:
pass_mode="multi-pass"



because randomized minibatches will not correspond to consecutive slices of
the label vector.

A label-aware randomized interface would conceptually support:



In [ ]:
for cell_ids, X_batch in atlas.get_minibatch_dense(...):
    y_batch = lookup_labels(cell_ids)
    train_step(X_batch, y_batch)



## When to Use Multi-pass Streams

Randomized multi-pass streams can already be used when the loss does not
require labels aligned from `obs`, for example:

- autoencoder reconstruction;
- selected unsupervised representation-learning objectives;
- clustering algorithms;
- custom iterative statistics;
- methods that generate targets directly from each `X_batch`.



In [ ]:
for X_batch in atlas.get_minibatch_dense(
    pass_mode="multi-pass",
    batch_size=2048,
    buffer_batch_num=5,
    max_batches=5000,
):
    train_unsupervised_step(X_batch)



Always set a finite stopping condition, such as `max_batches`, for a multi-pass
training loop.

## Limitations of This Example

This tutorial demonstrates how Atlas minibatches can be integrated into a
PyTorch training loop. It is not intended as a complete cell-type prediction
workflow.

It does not include:

- randomized supervised minibatches;
- class-imbalance correction;
- hyperparameter tuning;
- early stopping;
- regularization beyond the simple architecture;
- donor- or sample-aware validation;
- probability calibration;
- distributed or mixed-precision training.

These components should be selected according to the intended biological and
computational application.



## Close the Atlas

Close the database connection when this tutorial is complete. This releases
the DuckDB file lock so the same `.sasql` Atlas can be opened by another
notebook or Python session.


In [ ]:
atlas.close()


## Next Steps

See {doc}`apply-model-to-full-atlas` to apply the saved classifier across a
compatible Atlas read index without loading the complete expression matrix.

Continue with {doc}`implement-minibatch-kmeans` for an unsupervised iterative
method that naturally supports randomized multi-pass minibatches.